# Introduction to OpenAI API

## Environment Setup

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GEMINI_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTHROPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```

In [ ]:
import initialize_notebook # noqa

## Basics

In [ ]:
import os

import black
import openai

In [ ]:
client = openai.Client(
    api_key=os.environ.get("GEMINI_API_KEY", os.environ.get("GOOGLE_API_KEY")),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=10,  # Gemma 4 on Google's endpoint often fails with HTTP 500: retry.
)
model = "gemma-4-26b-a4b-it"

In [ ]:
history = [
    {"role": "user", "content": "Hello! My name is Vincent!"},
]

response = client.chat.completions.create(
    model=model,
    messages=history,
)
print(black.format_str(repr(response), mode=black.Mode(line_length=88)))

In [ ]:
print(response.choices[0].message.content)

## System/Developer Instructions

In [ ]:
system_instructions = """You are a cool Gen-Z assistant"""

In [ ]:
history = [
    {"role": "system", "content": system_instructions},
    {"role": "user", "content": "Hello! My name is Vincent!"},
]

response = client.chat.completions.create(
    model=model,
    messages=history,
)
print(response.choices[0].message.content)

## Structured Outputs

In [ ]:
system_instructions = \
"""Your goal is to extract events from a given email to be added to a calendar."""

In [ ]:
email = \
"""Hi everyone,
Quick rundown of what's coming up next week so you can block your calendars.
First, we have the quarterly review meeting on Monday, April 13 from 10:00 to 11:30 in the big conference room. Please come prepared with your Q1 numbers.
On Tuesday afternoon at 14:00, Marc from the Berlin office will give a 1-hour presentation on the new product roadmap. It'll be in room B204, but a Zoom link will also be sent out the morning of.
Don't forget the team lunch on Wednesday at 12:30 at Le Bistrot du Coin to welcome Julia, our new designer. I've already booked a table for 8.
Thursday is a bit busier: we have a client call with Acme Corp at 9:00 (should last about 30 minutes), and then the all-hands meeting from 16:00 to 17:00 in the auditorium.
Finally, a reminder that the office will be closed on Friday, April 17 for the public holiday — enjoy the long weekend!
Let me know if you have any conflicts.
Best,
Vincent"""

In [ ]:
history = [
    {"role": "system", "content": system_instructions},
    {"role": "user", "content": email},
]

response = client.chat.completions.create(
    model=model,
    messages=history,
)
print(response.choices[0].message.content)

In [ ]:
import datetime

from pydantic import dataclasses


@dataclasses.dataclass
class Event:
    title: str
    date: datetime.datetime
    location: str

@dataclasses.dataclass
class Events:
    events: list[Event]

In [ ]:
history = [
    {"role": "system", "content": system_instructions},
    {"role": "user", "content": email},
]

response = client.chat.completions.parse(
    # Gemma 4 does not reliably follow the JSON schema on Google's endpoint, so we use Gemini here.
    model="gemini-3.5-flash-lite",
    messages=history,
    response_format=Events,
)
print(black.format_str(repr(response), mode=black.Mode(line_length=88)))

In [ ]:
print(response.choices[0].message.content)

In [ ]:
events = response.choices[0].message.parsed
for event in events.events:
    print(event)

## Jinja Templating

In [ ]:
import datetime

import jinja2
import requests

In [ ]:
def get_location() -> str:
    """Returns the approximate location of the current machine based on its public IP address."""
    try:
        response = requests.get("https://ipinfo.io/json", timeout=10)
        response.raise_for_status()
        data = response.json()
        return f"{data['city']}, {data['region']}, {data['country']}"
    except (requests.RequestException, KeyError, ValueError):
        return "an unknown location"

location = get_location()
print(location)

In [ ]:
template_string = \
"""You are {{ assistant_name }}, a helpful assistant developed by {{ company_name }}.
The current date is {{ current_date.strftime('%A, %B %d %Y') }}.
The user's name is {{ user_name }} and they are located in {{ user_location }}.
"""
template = jinja2.Template(template_string, undefined=jinja2.StrictUndefined)
rendered_template = template.render(
    assistant_name="Jarvis",
    company_name="Stark Industries",
    current_date=datetime.datetime.now(tz=datetime.UTC).date(),
    user_name="Vincent",
    user_location=location,
)
print(rendered_template)